<a href="https://colab.research.google.com/github/Alirezas-12/fuel-distribution-project/blob/Colab/FuelDis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pulp

In [ ]:
import pulp


class FuelDistributionLP:
    """
    ماژول برنامه‌ریزی خطی (LP) برای ارسال بنزین از پالایشگاه‌ها
    به انبارهای نفت استانی از طریق خط لوله و ریل.
    """

    def __init__(self, refineries, depots):
        self.refineries = refineries      # لیست اسم پالایشگاه‌ها
        self.depots = depots              # لیست اسم انبارهای استانی
        self.supply = {}                  # {refinery: ظرفیت عرضه}
        self.demand = {}                  # {depot: تقاضا}
        self.cost = {}                    # {(refinery, depot, mode): هزینه هر واحد}
        self.capacity = {}                # {(refinery, depot, mode): سقف ظرفیت مسیر}
        self.model = None
        self.x = {}
        self.result = None

    def set_supply(self, supply_dict):
        self.supply = supply_dict

    def set_demand(self, demand_dict):
        self.demand = demand_dict

    def set_cost(self, cost_dict):
        # فقط مسیرهایی که اینجا هزینه‌شون تعریف بشه، در مدل مجاز خواهند بود
        self.cost = cost_dict

    def set_capacity(self, capacity_dict):
        self.capacity = capacity_dict

    def build_model(self):
        self.model = pulp.LpProblem("Fuel_Distribution", pulp.LpMinimize)
        routes = list(self.cost.keys())

        # یک متغیر تصمیم پیوسته برای هر مسیر مجاز
        self.x = {
            (i, j, m): pulp.LpVariable(f"x_{i}_{j}_{m}", lowBound=0)
            for (i, j, m) in routes
        }

        # تابع هدف: کمینه‌سازی هزینه کل حمل‌ونقل
        self.model += pulp.lpSum(
            self.cost[(i, j, m)] * self.x[(i, j, m)] for (i, j, m) in routes
        )

        # قید سقف عرضه هر پالایشگاه
        for i in self.refineries:
            self.model += (
                pulp.lpSum(self.x[(a, b, c)] for (a, b, c) in routes if a == i)
                <= self.supply[i],
                f"Supply_{i}",
            )

        # قید تأمین حداقل تقاضای هر انبار
        for j in self.depots:
            self.model += (
                pulp.lpSum(self.x[(a, b, c)] for (a, b, c) in routes if b == j)
                >= self.demand[j],
                f"Demand_{j}",
            )

        # قید سقف ظرفیت هر مسیر (در صورت تعریف)
        for (i, j, m) in routes:
            cap = self.capacity.get((i, j, m))
            if cap is not None:
                self.model += (self.x[(i, j, m)] <= cap, f"Cap_{i}_{j}_{m}")

    def solve(self):
        self.model.solve()
        self.result = {
            "status": pulp.LpStatus[self.model.status],
            "objective": pulp.value(self.model.objective),
            "shipments": {
                k: v.varValue for k, v in self.x.items() if v.varValue and v.varValue > 0
            },
            # ارزش سایه‌ای هر قید = مقدار دوگان آن (برای تحلیل حساسیت)
            "shadow_prices": {name: c.pi for name, c in self.model.constraints.items()},
        }
        return self.result

    def print_results(self):
        print("وضعیت حل:", self.result["status"])
        print("هزینه کل بهینه:", self.result["objective"])
        print("\nمقادیر ارسالی:")
        for (i, j, m), val in self.result["shipments"].items():
            print(f"  {i} -> {j} ({m}): {val}")
        print("\nارزش‌های سایه‌ای:")
        for name, val in self.result["shadow_prices"].items():
            print(f"  {name}: {val}")


In [ ]:
lp = FuelDistributionLP(refineries=["R1", "R2"], depots=["D1", "D2", "D3"])

lp.set_supply({"R1": 1000, "R2": 800})
lp.set_demand({"D1": 500, "D2": 600, "D3": 400})
lp.set_cost({
    ("R1", "D1", "pipeline"): 2,
    ("R1", "D2", "pipeline"): 3,
    ("R1", "D3", "rail"): 5,
    ("R2", "D1", "rail"): 4,
    ("R2", "D2", "pipeline"): 2,
    ("R2", "D3", "pipeline"): 3,
})
lp.set_capacity({route: 700 for route in lp.cost})

lp.build_model()
lp.solve()
lp.print_results()

وضعیت حل: Optimal
هزینه کل بهینه: 3600.0

مقادیر ارسالی:
  R1 -> D1 (pipeline): 500.0
  R1 -> D2 (pipeline): 200.0
  R2 -> D2 (pipeline): 400.0
  R2 -> D3 (pipeline): 400.0

ارزش‌های سایه‌ای:
  Supply_R1: 0.0
  Supply_R2: -1.0
  Demand_D1: 2.0
  Demand_D2: 3.0
  Demand_D3: 4.0
  Cap_R1_D1_pipeline: 0.0
  Cap_R1_D2_pipeline: 0.0
  Cap_R1_D3_rail: 0.0
  Cap_R2_D1_rail: 0.0
  Cap_R2_D2_pipeline: 0.0
  Cap_R2_D3_pipeline: 0.0


In [ ]:
import random


class FuelDistributionGA:
    """
    الگوریتم ژنتیک برای زمان‌بندی توزیع سوخت از انبار به جایگاه‌ها،
    با هدف کمینه‌سازی مجموع دیرکرد (tardiness) تحویل‌ها.
    """

    def __init__(self, stations, num_tankers, tanker_capacity):
        self.stations = stations              # لیست دیکشنری هر جایگاه: q, p, d, r
        self.n = len(stations)
        self.k = num_tankers
        self.capacity = tanker_capacity
        # کروموزوم = جایگشتی از ایندکس جایگاه‌ها + (k-1) ژن جداکننده منفی
        self.gene_pool = list(range(self.n)) + [-(i + 1) for i in range(self.k - 1)]

    def _decode(self, chromosome):
        # کروموزوم را بر اساس ژن‌های جداکننده به k مسیر تانکر می‌شکند
        routes = [[] for _ in range(self.k)]
        tanker = 0
        for gene in chromosome:
            if gene < 0:
                tanker += 1
            else:
                routes[tanker].append(gene)
        return routes

    def fitness(self, chromosome):
        # مجموع دیرکرد همه جایگاه‌ها + جریمه سنگین در صورت عبور از ظرفیت تانکر
        routes = self._decode(chromosome)
        total_tardiness = 0
        penalty = 0
        for route in routes:
            time = 0
            volume = 0
            for idx in route:
                st = self.stations[idx]
                start = max(time, st["r"])
                finish = start + st["p"]
                total_tardiness += max(0, finish - st["d"])
                time = finish
                volume += st["q"]
            if volume > self.capacity:
                penalty += (volume - self.capacity) * 1000
        return total_tardiness + penalty

    def _random_chromosome(self):
        genes = self.gene_pool[:]
        random.shuffle(genes)
        return genes

    def _tournament_select(self, population, fitnesses, k=3):
        contenders = random.sample(range(len(population)), k)
        best = min(contenders, key=lambda i: fitnesses[i])
        return population[best][:]

    def _order_crossover(self, parent1, parent2):
        # Order Crossover (OX): یک تکه از parent1 حفظ می‌شود، بقیه به ترتیب parent2 پر می‌شود
        size = len(parent1)
        a, b = sorted(random.sample(range(size), 2))
        child = [None] * size
        child[a:b] = parent1[a:b]
        fill_values = [g for g in parent2 if g not in child[a:b]]
        pos = 0
        for i in range(size):
            if child[i] is None:
                child[i] = fill_values[pos]
                pos += 1
        return child

    def _swap_mutation(self, chromosome, rate=0.1):
        # با احتمال rate، جای دو ژن تصادفی عوض می‌شود
        chromosome = chromosome[:]
        if random.random() < rate:
            i, j = random.sample(range(len(chromosome)), 2)
            chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
        return chromosome

    def run(self, population_size=40, generations=150, mutation_rate=0.15):
        population = [self._random_chromosome() for _ in range(population_size)]
        best_chromosome, best_fitness = None, float("inf")

        for _ in range(generations):
            fitnesses = [self.fitness(ind) for ind in population]
            gen_best_idx = min(range(len(population)), key=lambda i: fitnesses[i])
            if fitnesses[gen_best_idx] < best_fitness:
                best_fitness = fitnesses[gen_best_idx]
                best_chromosome = population[gen_best_idx][:]

            new_population = [best_chromosome[:]]  # نخبه‌گرایی (elitism)
            while len(new_population) < population_size:
                p1 = self._tournament_select(population, fitnesses)
                p2 = self._tournament_select(population, fitnesses)
                child = self._order_crossover(p1, p2)
                child = self._swap_mutation(child, mutation_rate)
                new_population.append(child)
            population = new_population

        return best_chromosome, best_fitness

    def explain(self, chromosome):
        # چاپ خوانای برنامه نهایی: کدوم تانکر، کدوم جایگاه، چه زمانی
        routes = self._decode(chromosome)
        for t, route in enumerate(routes):
            time = 0
            print(f"تانکر {t + 1}:")
            for idx in route:
                st = self.stations[idx]
                start = max(time, st["r"])
                finish = start + st["p"]
                late = max(0, finish - st["d"])
                print(f"  ایستگاه {idx}: شروع={start}, پایان={finish}, "
                      f"ددلاین={st['d']}, دیرکرد={late}")
                time = finish

    def get_schedule(self, chromosome):
        # نسخه ساختاریافته (لیست دیکشنری) برای استفاده در اپ/گزارش، به‌جای چاپ مستقیم
        routes = self._decode(chromosome)
        schedule = []
        for t, route in enumerate(routes):
            time = 0
            for idx in route:
                st = self.stations[idx]
                start = max(time, st["r"])
                finish = start + st["p"]
                late = max(0, finish - st["d"])
                schedule.append({
                    "tanker": t + 1,
                    "station_index": idx,
                    "start": start,
                    "finish": finish,
                    "deadline": st["d"],
                    "tardiness": late,
                })
                time = finish
        return schedule


In [ ]:
import random
random.seed(42)

stations = [
    {"q": 500, "p": 30, "d": 120, "r": 0},
    {"q": 300, "p": 20, "d": 90,  "r": 0},
    {"q": 400, "p": 25, "d": 150, "r": 0},
    {"q": 600, "p": 35, "d": 100, "r": 0},
    {"q": 200, "p": 15, "d": 200, "r": 0},
]

ga = FuelDistributionGA(stations=stations, num_tankers=2, tanker_capacity=1200)
best_chromo, best_fit = ga.run(population_size=40, generations=150, mutation_rate=0.15)

print("بهترین دیرکرد کل:", best_fit)
ga.explain(best_chromo)

بهترین دیرکرد کل: 0
تانکر 1:
  ایستگاه 4: شروع=0, پایان=15, ددلاین=200, دیرکرد=0
  ایستگاه 1: شروع=15, پایان=35, ددلاین=90, دیرکرد=0
  ایستگاه 0: شروع=35, پایان=65, ددلاین=120, دیرکرد=0
تانکر 2:
  ایستگاه 3: شروع=0, پایان=35, ددلاین=100, دیرکرد=0
  ایستگاه 2: شروع=35, پایان=60, ددلاین=150, دیرکرد=0
